In [23]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [24]:
import pandas as pd
import numpy as np

In [25]:
import sqlalchemy as sa
from sqlalchemy.orm import Session
from crmprtd import setup_logging
from pycds import Station, History

save_path = './comparison_forms/'

db_url = "postgresql://tongli1997@db.pcic.uvic.ca:5433/crmp?keepalives=1&keepalives_idle=300&keepalives_interval=300&keepalives_count=9&passfile=/workspaces/crmprtd/.pgpass"
log_file_path = save_path

engine = sa.create_engine(db_url, echo=False)
session = Session(engine)

session

In [26]:
path = '/workspaces/crmprtd/update_round2/input_tables/'
df = pd.read_excel(path + '20251223-Metadata-AllRequiredChanges_round2.xlsx', header = 1)   # pandas automatically uses openpyxl

# Select rows where ISSUE contains "Attribution" (case-sensitive)
df_attribution = df[df["ISSUE"].str.contains("Attribution", na=False)]

len(df_attribution)

df_attribution =  df_attribution[['ISSUE','Station ID', 'Network ID', 'Network Name', 'Native ID', 'Unique Names', 'Unique Location (String)']].reset_index(drop=True)
df_attribution["Network ID"] = pd.to_numeric(df_attribution["Network ID"], errors="coerce").astype("Int64")
df_attribution["Station ID"] = pd.to_numeric(df_attribution["Station ID"], errors="coerce").astype("Int64")
df_attribution


,ISSUE,Station ID,Network ID,Network Name,Native ID,Unique Names,Unique Location (String)
0,Attribution,2322,12,BC-FWx-> BC-TS,945,TS NAKA CREEK,"115.2144 W, 49.6673 N, Elev. 1051 m -> 126.433..."
1,Attribution,2633,9,BC-Air -> MVRD-AIR,E246240 -> T034,Abbotsford Airport - Walmsley Road,"122.343 W, 49.0235 N, Elev. 65 m"
2,Attribution,12610,9,BC-Air -> MVRD-AIR,New Westminster Sapperton Park -> T046,NA -> New Westminister,"122.894487 W, 49.227045 N, Elev. null m -> 122..."
3,Attribution,12608,9,BC-Air -> MVRD-AIR,Surrey East -> T015,NA -> Surrey East,"122.6942 W, 49.1328 N, Elev. null m -> 122.695..."
4,Attribution,12529,9,BC-Air -> MVRD-AIR,Abbotsford A Columbia Street -> T045,NA -> Abbotsford Airport,"122.3266 W, 49.0215 N, Elev. null m -> 122.326..."
...,...,...,...,...,...,...,...
131,Attribution,2571,14,BC-Snow -> BCH-GSO-HMP,4B15P,Lu Lake,"126.3 W, 54.2 N, Elev. 1310 m"
132,Attribution,2513,14,BC-Snow -> BCH-GSO-HMP,1A02P,McBride (Upper),"120.333 W, 53.3 N, Elev. 1620 m"
133,Attribution,2554,14,BC-Snow -> BCH-GSO-HMP,2F05P,Mission Creek,"118.95 W, 49.95 N, Elev. 1780 m"
134,Attribution,2543,14,BC-Snow -> BCH-GSO-HMP,2A21P,Molson Creek,"118.233 W, 52.2333 N, Elev. 1980 m"


In [27]:
df_attribution = df_attribution[['ISSUE','Station ID', 'Network ID', 'Network Name']]
df_attribution

def split_network_name(name):
    if pd.isna(name):
        return pd.Series([None, None, 0])

    parts = [p.strip() for p in name.split("->")]

    if len(parts) == 2:
        old_name, new_name = parts
        n_names = 2
    else:
        old_name = new_name = parts[0]
        n_names = 1

    return pd.Series([old_name, new_name, n_names])
    

df_attribution[['old_net_name', 'new_net_name', 'n_net_names']] = (
    df_attribution['Network Name'].apply(split_network_name)
)

df_attribution 

/tmp/ipykernel_11457/508565202.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_attribution[['old_net_name', 'new_net_name', 'n_net_names']] = (
/tmp/ipykernel_11457/508565202.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_attribution[['old_net_name', 'new_net_name', 'n_net_names']] = (


,ISSUE,Station ID,Network ID,Network Name,old_net_name,new_net_name,n_net_names
0,Attribution,2322,12,BC-FWx-> BC-TS,BC-FWx,BC-TS,2
1,Attribution,2633,9,BC-Air -> MVRD-AIR,BC-Air,MVRD-AIR,2
2,Attribution,12610,9,BC-Air -> MVRD-AIR,BC-Air,MVRD-AIR,2
3,Attribution,12608,9,BC-Air -> MVRD-AIR,BC-Air,MVRD-AIR,2
4,Attribution,12529,9,BC-Air -> MVRD-AIR,BC-Air,MVRD-AIR,2
...,...,...,...,...,...,...,...
131,Attribution,2571,14,BC-Snow -> BCH-GSO-HMP,BC-Snow,BCH-GSO-HMP,2
132,Attribution,2513,14,BC-Snow -> BCH-GSO-HMP,BC-Snow,BCH-GSO-HMP,2
133,Attribution,2554,14,BC-Snow -> BCH-GSO-HMP,BC-Snow,BCH-GSO-HMP,2
134,Attribution,2543,14,BC-Snow -> BCH-GSO-HMP,BC-Snow,BCH-GSO-HMP,2


In [28]:
df_attribution_2attri = df_attribution[df_attribution['n_net_names'] == 2]

df_attribution_2attri['new_net_name'] = (
    df_attribution_2attri['new_net_name']
    .replace(['MVRD-AIR', 'MVRD'], 'MVRD-AQ')
)

df_attribution_2attri = df_attribution_2attri.reset_index(drop=True)
df_attribution_2attri

,ISSUE,Station ID,Network ID,Network Name,old_net_name,new_net_name,n_net_names
0,Attribution,2322,12,BC-FWx-> BC-TS,BC-FWx,BC-TS,2
1,Attribution,2633,9,BC-Air -> MVRD-AIR,BC-Air,MVRD-AQ,2
2,Attribution,12610,9,BC-Air -> MVRD-AIR,BC-Air,MVRD-AQ,2
3,Attribution,12608,9,BC-Air -> MVRD-AIR,BC-Air,MVRD-AQ,2
4,Attribution,12529,9,BC-Air -> MVRD-AIR,BC-Air,MVRD-AQ,2
...,...,...,...,...,...,...,...
131,Attribution,2571,14,BC-Snow -> BCH-GSO-HMP,BC-Snow,BCH-GSO-HMP,2
132,Attribution,2513,14,BC-Snow -> BCH-GSO-HMP,BC-Snow,BCH-GSO-HMP,2
133,Attribution,2554,14,BC-Snow -> BCH-GSO-HMP,BC-Snow,BCH-GSO-HMP,2
134,Attribution,2543,14,BC-Snow -> BCH-GSO-HMP,BC-Snow,BCH-GSO-HMP,2


### select the netwok_id based on new_net_name, summarize back to the table 

In [29]:
# Create an empty column first
df_attribution_2attri['new_network_id'] = None

with engine.connect() as conn:
    for idx, row in df_attribution_2attri.iterrows():
        # Query network_id for this row's new_net_name
        res = conn.execute(
            sa.text("""
                SELECT network_id
                FROM meta_network
                WHERE network_display_name = :network_name
            """),
            {"network_name": row['new_net_name']}
        ).mappings().fetchone()

        if res is not None:
            df_attribution_2attri.at[idx, 'new_network_id'] = res['network_id']
        else:
            print(f"❌ Network '{row['new_net_name']}' not found for station {row['Station ID']}")

df_attribution_2attri

,ISSUE,Station ID,Network ID,Network Name,old_net_name,new_net_name,n_net_names,new_network_id
0,Attribution,2322,12,BC-FWx-> BC-TS,BC-FWx,BC-TS,2,71
1,Attribution,2633,9,BC-Air -> MVRD-AIR,BC-Air,MVRD-AQ,2,72
2,Attribution,12610,9,BC-Air -> MVRD-AIR,BC-Air,MVRD-AQ,2,72
3,Attribution,12608,9,BC-Air -> MVRD-AIR,BC-Air,MVRD-AQ,2,72
4,Attribution,12529,9,BC-Air -> MVRD-AIR,BC-Air,MVRD-AQ,2,72
...,...,...,...,...,...,...,...,...
131,Attribution,2571,14,BC-Snow -> BCH-GSO-HMP,BC-Snow,BCH-GSO-HMP,2,5
132,Attribution,2513,14,BC-Snow -> BCH-GSO-HMP,BC-Snow,BCH-GSO-HMP,2,5
133,Attribution,2554,14,BC-Snow -> BCH-GSO-HMP,BC-Snow,BCH-GSO-HMP,2,5
134,Attribution,2543,14,BC-Snow -> BCH-GSO-HMP,BC-Snow,BCH-GSO-HMP,2,5


In [30]:
check_sql = sa.text("""
SELECT
    s.station_id,
    n.network_id,
    n.network_display_name
FROM meta_network as n
JOIN meta_station as s ON n.network_id = s.network_id
WHERE s.station_id = :station_id
AND n.network_id = :network_id

""")


with engine.connect() as conn:
    for _, row in df_attribution_2attri.iterrows():

        res = conn.execute(
            check_sql,
            {
                "station_id": int(row["Station ID"]),
                "network_id": int(row["new_network_id"]),
            }
        ).fetchone()

        if res is None:
            print(f"❌ Station {row['Station ID']} not found")
            continue

        db_network_name = res[2]

        if db_network_name == row["new_net_name"]:
            print(
                f"✅ OK to update station {res.station_id}: "
                f"{db_network_name} → {row['new_net_name']}"
            )
        else:
            print(
                f"⚠️ Mismatch for station {res.station_id}: "
                f"DB={db_network_name}, expected={row['new_network_id']}"
            )

✅ OK to update station 2322: BC-TS → BC-TS
✅ OK to update station 2633: MVRD-AQ → MVRD-AQ
✅ OK to update station 12610: MVRD-AQ → MVRD-AQ
✅ OK to update station 12608: MVRD-AQ → MVRD-AQ
✅ OK to update station 12529: MVRD-AQ → MVRD-AQ
✅ OK to update station 12538: MVRD-AQ → MVRD-AQ
✅ OK to update station 12602: MVRD-AQ → MVRD-AQ
✅ OK to update station 12607: MVRD-AQ → MVRD-AQ
✅ OK to update station 12583: MVRD-AQ → MVRD-AQ
✅ OK to update station 2632: MVRD-AQ → MVRD-AQ
✅ OK to update station 1591: MVRD-AQ → MVRD-AQ
✅ OK to update station 1584: MVRD-AQ → MVRD-AQ
✅ OK to update station 2569: BCH-GSO-HMP → BCH-GSO-HMP
✅ OK to update station 2532: BCH-GSO-HMP → BCH-GSO-HMP
✅ OK to update station 2551: BCH-GSO-HMP → BCH-GSO-HMP
✅ OK to update station 2521: BCH-GSO-HMP → BCH-GSO-HMP
✅ OK to update station 2567: BCH-GSO-HMP → BCH-GSO-HMP
✅ OK to update station 2522: BCH-GSO-HMP → BCH-GSO-HMP
✅ OK to update station 2548: BCH-GSO-HMP → BCH-GSO-HMP
✅ OK to update station 2547: BCH-GSO-HMP → BCH-G

#### update the network id

In [31]:
# update_sql = sa.text("""
# UPDATE meta_station
# SET network_id = :new_network_id
# FROM meta_network AS n
# WHERE meta_station.station_id = :station_id
#   AND meta_station.network_id = n.network_id
# """)

# update_network_sql = sa.text("""
# UPDATE meta_network AS n
# SET network_display_name = :new_network_name
# WHERE n.network_id = :new_network_id
# """)

# with engine.begin() as conn:  # auto-commit / rollback
#     for _, row in df_attribution_2attri.iterrows():

#         # 1. Update the network_id in meta_station
#         result_station = conn.execute(
#             update_sql,
#             {
#                 "station_id": int(row["Station ID"]),
#                 "new_network_id": int(row["new_network_id"])
#             }
#         )

#         # 2. Update the network_name in meta_network
#         result_network = conn.execute(
#             update_network_sql,
#             {
#                 "new_network_id": int(row["new_network_id"]),
#                 "new_network_name": row["new_net_name"]
#             }
#         )

#         # ✅ Print checks
#         if result_station.rowcount == 1 and result_network.rowcount == 1:
#             print(
#                 f"✅ Station {row['Station ID']} updated: "
#                 f"network_id → {row['new_network_id']}, "
#                 f"network_name → {row['new_net_name']}"
#             )
#         else:
#             print(f"⚠️ Check station {row['Station ID']}, updates may have failed")

            


In [32]:
df_attribution_2attri[['Station ID', 'Network ID', 'new_network_id']]

,Station ID,Network ID,new_network_id
0,2322,12,71
1,2633,9,72
2,12610,9,72
3,12608,9,72
4,12529,9,72
...,...,...,...
131,2571,14,5
132,2513,14,5
133,2554,14,5
134,2543,14,5


In [33]:
df_attribution_2attri['new_network'] = df_attribution_2attri['new_network_id']> 50
df_attri_new_net = df_attribution_2attri[df_attribution_2attri['new_network'] == True]
df_attri_old_net = df_attribution_2attri[df_attribution_2attri['new_network'] == False]
df_attri_old_net = df_attri_old_net.drop(columns=['n_net_names'])
df_attri_old_net = df_attri_old_net.rename(columns={'Network ID': 'old_network_id'})

df_attri_new_net = df_attri_new_net.reset_index(drop=True)
df_attri_new_net

,ISSUE,Station ID,Network ID,Network Name,old_net_name,new_net_name,n_net_names,new_network_id,new_network
0,Attribution,2322,12,BC-FWx-> BC-TS,BC-FWx,BC-TS,2,71,True
1,Attribution,2633,9,BC-Air -> MVRD-AIR,BC-Air,MVRD-AQ,2,72,True
2,Attribution,12610,9,BC-Air -> MVRD-AIR,BC-Air,MVRD-AQ,2,72,True
3,Attribution,12608,9,BC-Air -> MVRD-AIR,BC-Air,MVRD-AQ,2,72,True
4,Attribution,12529,9,BC-Air -> MVRD-AIR,BC-Air,MVRD-AQ,2,72,True
...,...,...,...,...,...,...,...,...,...
80,"Location, Attribution",12453,12,BC-FWx-> BC-TS,BC-FWx,BC-TS,2,71,True
81,"Name, Location, Attribution",12458,12,BC-FWx-> BC-TS,BC-FWx,BC-TS,2,71,True
82,"Name, Location, Attribution",12526,12,BC-FWx-> BC-TS,BC-FWx,BC-TS,2,71,True
83,"Name, Location, Attribution",12957,12,BC-FWx-> BC-TS,BC-FWx,BC-TS,2,71,True


In [34]:
df_attri_old_net

,ISSUE,Station ID,old_network_id,Network Name,old_net_name,new_net_name,new_network_id,new_network
12,Attribution,2569,14,BC-Snow -> BCH-GSO-HMP,BC-Snow,BCH-GSO-HMP,5,False
13,Attribution,2532,14,BC-Snow -> BCH-GSO-HMP,BC-Snow,BCH-GSO-HMP,5,False
14,Attribution,2551,14,BC-Snow -> BCH-GSO-HMP,BC-Snow,BCH-GSO-HMP,5,False
15,Attribution,2521,14,BC-Snow -> BCH-GSO-HMP,BC-Snow,BCH-GSO-HMP,5,False
16,Attribution,2567,14,BC-Snow -> BCH-GSO-HMP,BC-Snow,BCH-GSO-HMP,5,False
17,Attribution,2522,14,BC-Snow -> BCH-GSO-HMP,BC-Snow,BCH-GSO-HMP,5,False
18,Attribution,2548,14,BC-Snow -> BCH-GSO-HMP,BC-Snow,BCH-GSO-HMP,5,False
19,Attribution,2547,14,BC-Snow -> BCH-GSO-HMP,BC-Snow,BCH-GSO-HMP,5,False
20,Attribution,2541,14,BC-Snow -> BCH-GSO-HMP,BC-Snow,BCH-GSO-HMP,5,False
21,Attribution,2565,14,BC-Snow -> BCH-GSO-HMP,BC-Snow,BCH-GSO-HMP,5,False


For one station:
	1.	Get all variables actually used by that station (old_station_vars)
	2.	Compare them against variables already defined in the new network
	3.	Split into two cases:
	•	✅ Variable already exists in new network → map old_vars_id → existing new vars_id
	•	❌ Variable does not exist → create it in new network → map old_vars_id → newly created vars_id
	4.	Build one vars_map
	5.	Bulk-update obs_raw.vars_id for the new station

In [35]:
from pycds import Variable
from sqlalchemy.orm import Session
from typing import Tuple, Dict, List
import pandas as pd

def create_vars_for_network(
    df_vars: pd.DataFrame,
    network_id: int,
    session: Session
) -> Tuple[List[int], Dict[int, int]]:
    """
    Create variables in meta_vars for a given network based on a DataFrame,
    and return the list of new variable IDs and a mapping from old_vars_id -> new_vars_id.
    
    Parameters
    ----------
    df_vars : pd.DataFrame
        DataFrame containing the variable info with columns:
        ['vars_id', 'unit', 'precision', 'standard_name', 'cell_method',
         'long_description', 'net_var_name', 'display_name', 'short_name']
    network_id : int
        The network_id to which these variables will belong.
    session : sqlalchemy.orm.Session
        The active SQLAlchemy session.
    
    Returns
    -------
    vars_created : List[int]
        List of newly created variable IDs.
    vars_map : Dict[int, int]
        Mapping from old_vars_id to new_vars_id.
    """
    vars_created = []
    vars_map = {}

    for _, row in df_vars.iterrows():
        v = Variable(
            network_id=network_id,
            unit=row["unit"],
            precision=row["precision"],
            standard_name=row["standard_name"],
            cell_method=row["cell_method"],
            description=row["long_description"],
            name=row["net_var_name"],
            display_name=row["display_name"],
            short_name=row["short_name"],
        )

        session.add(v)
        session.flush()  # assigns vars_id immediately

        vars_created.append(v.id)
        vars_map[row["vars_id"]] = v.id

    session.commit()
    print(f"✅ Created {len(vars_created)} variables for network {network_id}")

    return vars_created, vars_map

In [36]:
import json
from sqlalchemy import text
from sqlalchemy.engine import Engine
from typing import Dict


SQL_UPDATE_VARS_BULK = text("""
UPDATE obs_raw o
SET vars_id = m.new_vars_id
FROM meta_history h
JOIN meta_station s
  ON h.station_id = s.station_id
CROSS JOIN json_to_recordset(:pairs_json)
     AS m(old_vars_id INT, new_vars_id INT)
WHERE o.history_id = h.history_id
  AND s.station_id = :station_id
  AND o.vars_id = m.old_vars_id
""")


def remap_obs_vars_for_station(
    engine: Engine,
    station_id: int,
    vars_map: Dict[int, int],
) -> int:
    """
    Bulk-remap obs_raw.vars_id for a given station using an old→new vars_id map.

    Parameters
    ----------
    engine : sqlalchemy.engine.Engine
        SQLAlchemy engine.
    station_id : int
        Station ID whose observations will be updated.
    vars_map : dict[int, int]
        Mapping of old_vars_id → new_vars_id.

    Returns
    -------
    int
        Number of rows updated.
    """
    if not vars_map:
        print("⚠️ vars_map is empty — nothing to update.")
        return 0


    # Only include old_vars_id that differ from current obs_raw
    rows_to_update = [
        (old, new) for old, new in vars_map.items()
        if old != new
    ]

    if not rows_to_update:
        print("🔹 All vars already mapped — nothing to update")
        return 0

    rows_to_update_json = json.dumps(
        [{"old_vars_id": old, "new_vars_id": new} for old, new in rows_to_update]
    )


    # Ensure plain Python int (avoid numpy.int64 issues)
    station_id = int(station_id)

    with engine.begin() as conn:
        res = conn.execute(
            SQL_UPDATE_VARS_BULK,
            {
                "pairs_json": rows_to_update_json,
                "station_id": station_id,
            }
        )

    print(
        f"✅ vars_id remap completed | "
        f"station={station_id} | "
        f"mappings={len(vars_map)} | "
        f"rows_updated={res.rowcount}"
    )

    return res.rowcount

In [37]:

from sqlalchemy import text

SQL_VARS_INFO = text("""
SELECT DISTINCT
       -- o.vars_id,
       v.*,
       -- m.station_id,
       s.native_id
       -- m.station_name,
       -- s.network_id AS station_network_id
FROM meta_history AS m
JOIN meta_station AS s ON m.station_id = s.station_id
JOIN obs_raw AS o ON m.history_id = o.history_id
JOIN meta_vars AS v ON o.vars_id = v.vars_id
WHERE s.station_id = :station_id
""")

SQL_VARS_new = text("""
SELECT DISTINCT
       v.vars_id,
       v.unit,
       v.precision,
       v.standard_name,
       v.cell_method,
       v.long_description,
       v.net_var_name,
       v.display_name,
       v.short_name
FROM meta_vars v
JOIN meta_network n ON v.network_id = n.network_id
WHERE n.network_id = :network_id
""")

def sync_station_vars(
    row,
    engine,
    session,
    dry_run=False,
):
    station_id = int(row["Station ID"])
    old_net_id = int(row["Network ID"])
    new_net_id = int(row["new_network_id"])

    print(f"\n🚉 Processing station {station_id} ({old_net_id} → {new_net_id})")

    # --- load vars used by this station ---
    old_station_vars = pd.read_sql(
        SQL_VARS_INFO,
        engine,
        params={"station_id": station_id}
    )

    if old_station_vars.empty:
        print("⚠️ No obs / vars found for this station — skipping.")
        return 0

    # --- load vars available in target network ---
    new_net_vars = pd.read_sql(
        SQL_VARS_new,
        engine,
        params={"network_id": new_net_id}
    )

    VAR_MATCH_COLS = [
        "unit",
        "precision",
        "standard_name",
        "cell_method",
        "long_description",
        "net_var_name",
        "display_name",
        "short_name",
    ]

    df_compare = old_station_vars.merge(
        new_net_vars[VAR_MATCH_COLS + ["vars_id"]],
        on=VAR_MATCH_COLS,
        how="left",
        suffixes=("", "_new_net"),
        indicator=True
    )

    df_existing = df_compare[df_compare["_merge"] == "both"].copy()
    df_missing  = df_compare[df_compare["_merge"] != "both"].copy()

    print(
        f"📊 Vars summary:\n"
        f"  ✔ existing: {len(df_existing)}\n"
        f"  ➕ missing : {len(df_missing)}"
    )

    # --- build vars_map ---
    vars_map = {
        int(r["vars_id"]): int(r["vars_id_new_net"])
        for _, r in df_existing.iterrows()
    }

    # --- create missing vars if needed ---
    if not df_missing.empty:
        if dry_run:
            print("🧪 DRY-RUN: would create missing variables")
        else:
            _, new_vars_map = create_vars_for_network(
                df_missing,
                network_id=new_net_id,
                session=session
            )
            vars_map.update(new_vars_map)

    # --- safety check ---
    assert df_compare["vars_id"].nunique() == len(vars_map), \
        "❌ vars_map incomplete!"

    if not vars_map:
        print("⚠️ Nothing to remap — skipping obs update.")
        return 0

    # --- remap obs ---
    if dry_run:
        print(f"🧪 DRY-RUN: would remap {len(vars_map)} vars")
        return 0

    rows_updated = remap_obs_vars_for_station(
        engine=engine,
        station_id=station_id,
        vars_map=vars_map,
    )

    print(f"✅ Updated {rows_updated} obs rows")

    return rows_updated


In [38]:
total_rows = 0

for i, row in df_attri_new_net.iterrows():
    print(f"\n[{i+1}/{len(df_attri_new_net)}]")

    try:
        n = sync_station_vars(
            row=row,
            engine=engine,
            session=session,
            dry_run=False,   # set True to preview
        )
        total_rows += n

    except Exception as e:
        print(f"❌ Failed at station {row['Station ID']}: {e}")
        session.rollback()


[1/85]

🚉 Processing station 2322 (12 → 71)
📊 Vars summary:
  ✔ existing: 0
  ➕ missing : 13
✅ Created 13 variables for network 71
✅ vars_id remap completed | station=2322 | mappings=13 | rows_updated=1127521
✅ Updated 1127521 obs rows

[2/85]

🚉 Processing station 2633 (9 → 72)
📊 Vars summary:
  ✔ existing: 0
  ➕ missing : 2
✅ Created 2 variables for network 72
✅ vars_id remap completed | station=2633 | mappings=2 | rows_updated=132524
✅ Updated 132524 obs rows

[3/85]

🚉 Processing station 12610 (9 → 72)
📊 Vars summary:
  ✔ existing: 0
  ➕ missing : 1
✅ Created 1 variables for network 72
✅ vars_id remap completed | station=12610 | mappings=1 | rows_updated=2490
✅ Updated 2490 obs rows

[4/85]

🚉 Processing station 12608 (9 → 72)
📊 Vars summary:
  ✔ existing: 1
  ➕ missing : 0
✅ vars_id remap completed | station=12608 | mappings=1 | rows_updated=2826
✅ Updated 2826 obs rows

[5/85]

🚉 Processing station 12529 (9 → 72)
📊 Vars summary:
  ✔ existing: 1
  ➕ missing : 1
✅ Created 1 variab